# Job Listing Prioritization for Career Changers: Profiles (career switches)

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

Finds **real career changers** in a public resume dataset: people who moved from another background into a **computer and mathematical** role. Each resume is read **as a whole**: jobs and education in one timeline, in date order, each entry labelled tech/math, other field, unclear, or skip (internships, school jobs, high school). The person's **original background** is everything before their first tech or math entry. They're a career changer if that background includes another field. **Only people who retrained count as candidates** (decided 2026-10-09): another field first, then a tech or math program (a degree, master's, bootcamp or certificate), then the tech job. Two pools:
- **`candidate: retrained, new grad from another field`:** studied another field (for example English or biology), then trained into tech or math, then the tech job.
- **`candidate: retrained, after working in another field`:** worked in another field, then trained into tech or math, then the tech job.

People who switched straight into tech with no tech or math training are kept out (`changed jobs without tech or math training`, `new grad from another field without tech or math training`); `keep_direct_switches` in the rules brings them back.

Someone whose timeline *starts* in tech or math (a computer science degree, then a developer job) isn't changing background, so they're rejected. Each one gives us what the transferable-fit model (Option B, see `docs/fit_analysis.md`) and role discovery need:
- **A pre-switch profile:** the jobs and education *before* the switch. This is the career changer.
- **A real label:** the role they actually moved into, the "good fit" the model should find.

A career changer is **anyone switching from their current background to a different role or field**.

**How we get the rules right: a review loop.**
1. `src/career_switch.py` holds every rule (`RULES`) and its version (`RULES_VERSION`).
2. Section 4 draws a reproducible review sample and writes a blank review file.
3. Jacob marks each person good, bad or unsure, with a reason code.
4. Section 5 turns the marks into counts and precision. Those aggregates (never rows or notes) are what's shared when asking for rule changes.
5. Change the rules, bump `RULES_VERSION`, log it in section 6, and run a new round.

**Done when** each candidate pool (the two retrained pools) reaches **90% precision or higher in two rounds in a row** (30 people each), and samples from the rejected pools turn up few good people (so we aren't throwing real switches away). Then section 7 freezes the rules and saves the table for the ML step.

**Data:** Kaggle `suriyaganesh/resume-dataset-structured` (MIT license, version 2 of 2024-11-14): about 54,000 resumes, converted from PDFs by OCR and an LLM, so expect extraction errors. Saved locally under `data/external/resume_structured/` (git-ignored).

**Data rules:**
- **It holds personal details.** `people.csv` (names, email, phone, LinkedIn) is never loaded. From `experience.csv` and `education.csv`, only titles, programs and dates are loaded, never employers or locations.
- Aggregates only, except the review cell in 4b, tagged `data-rows`: clear its output before committing or exporting.
- Never paste rows, titles from the review sample, or review notes into AI tools. Share only the section 5 counts.

### Clarifying questions
- How many people in the dataset switched into a computer and mathematical role, and from which backgrounds?
- Which roles do they switch into (data, IT support, software, QA, …)? This is the role-discovery target list.
- How long were they in their old background, and how long was the gap before the switch?
- How often do the rules mistake a title change, a student job or an already-technical background for a career change?
- For new grads: how often is the field of study missing or too vague ("Bachelor of Science") to tell?

### Leakage rule
The pre-switch profile uses **only jobs and education that started before the switch month**. Programs with no start date are placed by degree level: bachelor's and below at the start of the timeline, master's, certificates and bootcamps just before the switch. People who rely on this are flagged in `study_undated`. **Skills aren't dated** (the dataset extracts them from all work experience), so a person's skill list may include skills learned *after* the switch. Skills stay out of the pre-switch profile until we decide how to handle them.

### Roadmap
1. Setup
2. Load (only the columns we need), then remove duplicate resumes (2b)
3. Dates and title flags
4. Find the switches, viable candidates at a glance (4a.1), career changers who retrained into tech or math (4a.2), then the review round
5. Review summary
6. Rules log
7. Freeze and save (once the rules pass)

## 1. Setup

In [ ]:
# 1a Libraries, shared helpers and the career-switch rules
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from scipy import stats
from sklearn.linear_model import LogisticRegression

from src.eda_helpers import BLUE, DATA_DIR, INK_2, MUTED, ORANGE, bar_h, finish, heatmap, show, use_chart_style
from src.career_switch import (REASONS, RULES, RULES_VERSION, background_area, find_switches, person_timeline,
                               prepare_education, review_summary, tag_jobs)
use_chart_style()

RESUME_DIR = DATA_DIR / "external" / "resume_structured"
REVIEW_DIR = DATA_DIR / "profiles" / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
print("rules version:", RULES_VERSION)

## 2. Load

Only the columns the rules need. If the dataset's headers differ from its documentation, this stops and lists what's missing.

In [ ]:
# 2a Find the files and load only the columns we need
import re
import zipfile

def is_named(path, name):
    """Match a file name, ignoring case and a leading number prefix (the zip has 04_experience.csv)."""
    return re.sub(r"^\d+_", "", Path(path).name.lower()) == name.lower()

def unzip_if_needed():
    """If the CSVs aren't unzipped yet, extract the Kaggle zip found anywhere under data/ (it must contain experience.csv)."""
    if any(is_named(p, "experience.csv") for p in RESUME_DIR.rglob("*")):
        return
    zips = sorted(DATA_DIR.rglob("*.zip"))
    for zp in zips:
        with zipfile.ZipFile(zp) as z:
            names = z.namelist()
            if not any(is_named(n, "experience.csv") for n in names):
                continue
            if any(Path(n).is_absolute() or ".." in Path(n).parts for n in names):
                raise ValueError(f"{zp.name} has unsafe paths; not extracting")
            RESUME_DIR.mkdir(parents=True, exist_ok=True)
            z.extractall(RESUME_DIR)
        print(f"unzipped {zp.relative_to(DATA_DIR.parent)} into {RESUME_DIR.relative_to(DATA_DIR.parent)} ({len(names)} files); you can delete the zip")
        return
    # Nothing matched: report file names only (never contents), so the problem can be fixed
    print("No zip under data/ contains experience.csv. Zips found:", [str(z.relative_to(DATA_DIR.parent)) for z in zips] or "none")
    for zp in zips:
        try:
            with zipfile.ZipFile(zp) as z:
                print(f"  {zp.name}: {len(z.namelist())} files; CSVs: {[Path(n).name for n in z.namelist() if n.lower().endswith('.csv')][:20]}")
        except zipfile.BadZipFile:
            print(f"  {zp.name}: not a valid zip (still downloading, or the download failed)")

def find_csv(name):
    unzip_if_needed()
    hits = sorted(p for p in RESUME_DIR.rglob("*") if is_named(p, name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {RESUME_DIR}. Download the dataset first (see docs/fit_analysis.md).")
    return hits[0]

def load(name, cols):
    path = find_csv(name)
    header = pd.read_csv(path, nrows=0).columns
    missing = [c for c in cols if c not in header]
    if missing:
        raise KeyError(f"{name}: missing columns {missing}; found {list(header)}")
    return pd.read_csv(path, usecols=cols, dtype=str, encoding_errors="replace")

experience = load("experience.csv", ["person_id", "title", "start_date", "end_date"])
education = load("education.csv", ["person_id", "program", "start_date"])

show("Loaded tables (shape, and the share of each column filled)", pd.DataFrame({
    name: {"rows": len(df), "people": df.person_id.nunique(),
           **{f"{c} filled": round(df[c].notna().mean(), 3) for c in df.columns if c != "person_id"}}
    for name, df in {"experience": experience, "education": education}.items()
}).T)

### 2b Duplicates

The dataset combines several resume collections, and the first run's counts were almost all multiples of 3, which suggests each resume appears about three times under different `person_id`s. Duplicates would show the same person twice in a review round and put copies of one person on both sides of a train/test split (leakage).

Each person gets a **fingerprint**: every job (title, start and end date as written) plus every program (name and start date), sorted. People with the same fingerprint are the same resume; the lowest `person_id` is kept. Only **exact** copies are caught: a copy with an OCR difference gets a different fingerprint.

In [ ]:
# 2b Fingerprint every resume and keep one copy of each
from src.career_switch import fingerprints

DEDUPE = True
fp = fingerprints(experience, education)
N_BEFORE_DEDUPE = len(fp)
if DEDUPE:
    order = np.argsort(pd.to_numeric(fp.index, errors="coerce"), kind="stable")
    keep = fp.iloc[order].drop_duplicates().index  # the lowest person_id of each fingerprint
    experience = experience[experience.person_id.isin(keep)]
    education = education[education.person_id.isin(keep)]

show("Duplicate resumes", pd.Series({
    "resumes in the dataset": N_BEFORE_DEDUPE,
    "duplicates removed": N_BEFORE_DEDUPE - experience.person_id.nunique(),
    "resumes left": experience.person_id.nunique(),
}).to_frame("resumes"))

**What the data shows**
- *To fill in after running: how many resumes were duplicates, and how many are left.*

## 3. Dates and title flags

Dates become months (`MM/YYYY`; a bare year becomes January; "Present" becomes the dataset's update month, 2024-11). Titles are matched against the rules: which computer and mathematical bucket they fall in, and whether they're senior, school or internship work, or already technical.

In [ ]:
# 3a Parse dates and flag titles
jobs = tag_jobs(experience)
show("Computer and mathematical jobs by bucket", jobs.cm_bucket.value_counts().to_frame("jobs"))

## 4. Find the switches

For each person, in date order: their **first computer and mathematical job** is the switch, and the non-computer-and-mathematical jobs before it are their **background**. Everything before it is read as one timeline with their education. Their **original background** is everything before their first tech or math entry (a tech job, or a tech or math program). A person is a **candidate** if that background includes another field **and** they trained into tech or math before the switch: `candidate: retrained, new grad from another field` or `candidate: retrained, after working in another field`. Each rejection is kept with its reason, so we can review the rejected pools too.

In [ ]:
# 4a One row per person: candidate, or the first rule that rejected them
switches = find_switches(jobs, education)
funnel = switches.status.value_counts().rename_axis("status").to_frame("people")
funnel["share"] = (funnel.people / len(switches)).round(3)
show(f"People by status (rules version {RULES_VERSION})", funnel)

fig, ax = plt.subplots(figsize=(9, 4.2))
bar_h(ax, funnel.people, f"People by status, rules version {RULES_VERSION}", "people")
finish()

**What the data shows**
- *To fill in after running: how many candidates, which rule rejects the most people, the largest target buckets, and the typical background length and gap.*

### 4a.1 Viable candidates

How many people the rules leave, and who they are in numbers. **Viable** means they passed the rules, not that they've been reviewed, so the counts are upper bounds until the review rounds. Counts are of unique people (after 2b). Aggregates only.

In [ ]:
# 4a.1a How many: from every resume to viable candidates
POOLS = ["candidate: retrained, new grad from another field", "candidate: retrained, after working in another field"]
viable = switches[switches.status.isin(POOLS)].copy()
viable["path"] = viable.status.str.removeprefix("candidate: retrained, ")
viable["background_area"] = np.where(viable.origin_type.eq("job"), viable.source_title, viable.source_study)
viable["background_area"] = viable.background_area.map(background_area)
viable["switch_year"] = viable.switch_month.map(lambda m: m.year if pd.notna(m) else np.nan)
viable["years_worked_before"] = viable.background_months / 12
viable["programs"] = education.groupby("person_id").size().reindex(viable.index, fill_value=0)

steps = pd.Series({
    "resumes in the dataset": N_BEFORE_DEDUPE,
    "unique people": len(switches),
    "with a readable job date": (switches.status != "no dated jobs").sum(),
    "ever held a computer and mathematical job": switches.target_bucket.notna().sum(),
    "first such job not senior": (switches.target_bucket.notna() & (switches.status != "first computer and mathematical role is senior")).sum(),
    "viable: retrained into tech or math (total)": len(viable),
    "  new grad from another field": (viable.path == "new grad from another field").sum(),
    "  after working in another field": (viable.path == "after working in another field").sum(),
})
show(f"From every resume to viable candidates (rules version {RULES_VERSION})",
     pd.DataFrame({"people": steps, "share of unique people": (steps / len(switches)).round(3)}))

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(steps.index[::-1].str.strip(), steps.values[::-1], color=[ORANGE if s.startswith("  ") else BLUE for s in steps.index[::-1]], height=0.65)
for y, v in enumerate(steps.values[::-1]):
    ax.text(v, y, f" {v:,}", va="center", color=INK_2, fontsize=8.5)
ax.set(title="From every resume to viable candidates", xlabel="people (orange: the two paths)")
ax.grid(axis="y", visible=False)
ax.margins(x=0.15)
finish()

**What it shows**
- *To fill in after running: how many viable candidates there are in total and on each path, and the biggest drop in the funnel.*

In [ ]:
# 4a.1b Who they are, in numbers: summary statistics, distributions and correlations
METRICS = {
    "switch_year": "switch year",
    "years_worked_before": "years worked before",
    "n_jobs_before": "jobs before the switch",
    "n_jobs": "jobs on the resume",
    "programs": "programs on the resume",
}
m = viable[list(METRICS)].rename(columns=METRICS).astype(float)
gap = viable.loc[viable.origin_type.eq("job"), "gap_months"].rename("gap before the switch, months (after-work path)")
both = pd.concat([m, gap], axis=1)
summary = both.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T
summary["skew"] = both.skew()
show("Summary statistics, all viable candidates", summary.round(1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={"width_ratios": [1, 1, 1.1]})
for ax, col, subset, note in [(axes[0], "switch year", m["switch year"], ""),
                              (axes[1], "years worked before", m["years worked before"][m["years worked before"] > 0], "\n(only people who worked before)")]:
    x = subset.dropna()
    counts, edges, _ = ax.hist(x, bins="auto", color=BLUE, alpha=0.85)
    grid = np.linspace(edges[0], edges[-1], 200)
    ax.plot(grid, stats.norm.pdf(grid, x.mean(), x.std()) * len(x) * np.diff(edges).mean(), color=ORANGE,
            label="bell curve with the same mean and std")
    ax.set(title=f"{col.capitalize()} (n = {len(x):,}, skew {x.skew():.1f}){note}", xlabel=col, ylabel="people")
    ax.legend(fontsize=8)
heatmap(axes[2], m.loc[:, m.std() > 0].corr(method="spearman"), "Spearman correlation", fmt="{:.2f}", cmap="RdBu_r", vmin=-1, vmax=1)
finish()

**What it shows**
- *To fill in after running: typical (median) switch year and years worked before; how far each histogram is from the bell curve (skew above about 1 means a long right tail, so medians describe it better than means); and any correlation stronger than ±0.3.*

### 4a.2 Retrained into tech or math: what they trained into, and where it led

One row per viable candidate, stating the **field they trained into** (grouped by `TRAINING_FIELDS` in `src/career_switch.py`) and the program as written. The full table is saved to `data/profiles/` for browsing; everything printed here is aggregate except the preview in 4a.2d.

In [ ]:
# 4a.2a The retrained table, saved, with totals by path
RETRAINED_COLS = {
    "path": "path",
    "background_area": "original background (area)",
    "source_title": "main background job",
    "source_study": "field studied first",
    "retrain_field": "field trained into (grouped)",
    "retrain_study": "program trained in",
    "target_bucket": "first tech role (bucket)",
    "target_title": "first tech job title",
    "switch_period": "when they switched",
    "background_months": "months worked in another field",
    "study_undated": "relied on an undated program",
}
FIELD, ROLE, AREA, WHEN = "field trained into (grouped)", "first tech role (bucket)", "original background (area)", "when they switched"
retrained = viable[list(RETRAINED_COLS)].rename(columns=RETRAINED_COLS).sort_values([FIELD, ROLE])
RETRAINED_PATH = DATA_DIR / "profiles" / f"retrained_candidates_v{RULES_VERSION}.csv"
retrained.to_csv(RETRAINED_PATH)
print(f"saved {len(retrained):,} rows to {RETRAINED_PATH.relative_to(DATA_DIR.parent)}")

by_path = retrained.path.value_counts()
show("Retrained career changers by path", pd.concat([by_path, pd.Series({"total": by_path.sum()})])
     .rename_axis("path").to_frame("people").assign(share=lambda d: (d.people / by_path.sum()).round(3)))

In [ ]:
# 4a.2b What they trained into, where they came from, and where it led
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
bar_h(axes[0], retrained[FIELD].value_counts(), "Field trained into", "people")
bar_h(axes[1], retrained[AREA].value_counts().head(10), "Original background (top 10 areas)", "people", color=MUTED)
bar_h(axes[2], retrained[ROLE].value_counts(), "First tech role", "people", color=ORANGE)
finish()

programs = retrained["program trained in"].dropna().str.lower().str.strip().value_counts()
show("Most common programs trained in, as written (top 10, shared by at least 5 people): checks the grouping",
     programs[programs >= 5].head(10).rename_axis("program").to_frame("people"))

**What it shows**
- *To fill in after running: the top fields trained into, the top backgrounds, the top first tech roles, Flag any program in the top 10 that's in the wrong group.*

In [ ]:
# 4a.2c When: career-change dates by field, and whether the path changed over time
PERIODS = ["before 2000", "2000-2004", "2005-2009", "2010-2014", "2015-2019", "2020 or later", "not dated"]
when = pd.crosstab(retrained[FIELD], retrained[WHEN]).reindex(columns=PERIODS, fill_value=0)
when = when.loc[:, when.sum() > 0]  # drop empty periods
when = when.loc[when.sum(axis=1).sort_values(ascending=False).index]

fig, axes = plt.subplots(1, 2, figsize=(17, 0.42 * len(when) + 2.2), gridspec_kw={"width_ratios": [1.3, 1]})
heatmap(axes[0], when, "Field trained into × when they switched (people)")

d = viable[["switch_year"]].assign(new_grad=viable.path.eq("new grad from another field").astype(int)).dropna()
yearly = d.groupby("switch_year").new_grad.agg(["mean", "size"])
axes[1].scatter(yearly.index, yearly["mean"], s=15 + 185 * yearly["size"] / yearly["size"].max(),
                color=BLUE, alpha=0.7, label="share each year (dot size = people)")
if d.new_grad.nunique() == 2:
    logit = LogisticRegression().fit(d[["switch_year"]] - 2010, d.new_grad)
    years = np.arange(d.switch_year.min(), d.switch_year.max() + 1)
    axes[1].plot(years, logit.predict_proba(pd.DataFrame({"switch_year": years - 2010}))[:, 1], color=ORANGE,
                 label=f"logistic fit: odds × {np.exp(5 * logit.coef_[0][0]):.2f} every 5 years")
axes[1].set(title="Share who retrained as new grads (vs. after working), by switch year",
            xlabel="year of the switch", ylabel="share new grads", ylim=(-0.02, 1.02))
axes[1].legend(fontsize=8)
finish()

**What it shows**
- *To fill in after running: which periods most switches come from, whether some fields are recent (for example data science), and whether retraining shifted toward new grads or toward people who'd already worked (odds above 1 every 5 years means more new grads over time). Switch date is descriptive only; it never filters anyone.*

In [ ]:
# 4a.2d Preview: the first rows of the retrained table (shows rows: clear before committing)
display(retrained.head(20))

### 4b Review round

Set `ROUND` and `POOL`, then run the next cell. `POOL` is one of the two candidate statuses to check precision, or a rejection status from the table above (for example `"background already in tech"`) to check whether a rule throws real switches away. The sample is reproducible (`random_state = ROUND`) and skips anyone already reviewed.

The cell shows each person's **whole timeline** (job titles and programs with dates; no employers or locations), with each entry's label and the switch marked, and writes a blank review file to `data/profiles/review/`. It never overwrites a file you've started filling in.

**How to review:** open the file and, for each person, fill in:
- `verdict`: `good`, `bad` or `unsure`
- `reason`: one of the codes below (`good` for good ones)
- `note` (optional): what the rules missed, in your own words. Notes stay local and are never shared.

In [ ]:
# 4b Review settings
ROUND = 1
POOL = "candidate: retrained, new grad from another field"  # or "candidate: retrained, after working in another field"
N = 30
show("Reason codes", pd.Series(REASONS, name="meaning").to_frame())

## 5. Review summary

Counts and precision from every review file. **This output is aggregates only, safe to share** when asking for rule changes. Notes are never included. Precision = good ÷ (good + bad).

In [ ]:
# 5a Precision by round and pool; reasons for the bad ones
files = sorted(REVIEW_DIR.glob("round_*.csv"))
if not files:
    print("No review files yet: run 4c first.")
else:
    reviews = pd.concat([pd.read_csv(p, dtype=str).fillna("") for p in files])
    bad_codes = set(reviews.reason[reviews.verdict.eq("bad")]) - set(REASONS)
    if bad_codes:
        print("Unknown reason codes (fix them in the review file):", sorted(bad_codes))
    by_round, reasons = review_summary(reviews)
    show("Review results by round and pool", by_round)
    show("Why the bad ones were bad", reasons)

## 6. Rules log

One row per rules version: what changed and why, and the review result that prompted it.

| Version | Change | Why | Candidate precision (round) |
|:--|:--|:--|:--|
| 1 | First rules: first computer and mathematical job = the switch; reject senior first roles, school-only backgrounds, technical backgrounds, and backgrounds under 12 months | Strict start: precision first, then check the rejected pools for real switches | *not reviewed* |
| 2 | **New grads:** people with no real job before the switch are judged on what they studied. Candidate if they studied a field other than tech or math; rejected if they studied tech or math, only high school, or the field is unclear. Job switchers who studied tech are flagged (`studied_tech`), not rejected. Candidate statuses split into `candidate: changed jobs` and `candidate: new grad from another field` | Jacob: version 1 counted many new grads; a new grad only counts if they studied a different background, not math or tech | *not reviewed* |
| 3 | **Whole-resume timeline:** jobs and education merged in date order and labelled; the original background is everything before the first tech or math entry. Career changer if it includes another field. New pool `candidate: retrained into tech or math` (another field, then a tech or math program, then the tech job). Undated programs placed by degree level. Technical-sounding job titles count as tech entries instead of rejecting the person outright | Jacob: people may study something unrelated first, then study again into math or tech; read resumes holistically | *not reviewed* |
| 4 | **Only retrained people are candidates**, in two pools: `candidate: retrained, new grad from another field` and `candidate: retrained, after working in another field`. Direct switches with no tech or math training get non-candidate statuses (`keep_direct_switches` = False). Also: duplicate resumes removed before the rules run (2b) | Jacob: we only care about career changers who retrained into tech or math | *to fill in after round 1* |

## 7. Freeze and save

*Once the rules pass (90%+ precision two rounds in a row):* save the candidates with their pre-switch profiles (jobs and education before the switch month only) to `data/profiles/switchers_v<RULES_VERSION>.pkl` for the ML step, and log the decision in `docs/decisions.md`.